# 33 · Optimización de hiperparámetros de E3·S4_s2 con la matriz C3 (HU0007-3)

Encargo `docs/modeling/fe_hiperparametros.md`, fase 3. Todo lo que decide está preregistrado y commiteado antes
del estudio en `src/modeling/hiperparametros.py → PREREGISTRO_FASE3`. El protocolo (`protocolo_arquitectura.py`)
no cambia. Este notebook solo orquesta; nada se escribe en `data/`.

**Límite:** la búsqueda usa las mismas temporadas de selección (2022–2023), así que las métricas de la candidata
son optimistas. 2024 es solo de contraste.

In [1]:
import sys, json, math, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import optuna
from src.eda.carga import cargar_integrado
from src.modeling import cache_experimentos as cache
from src.modeling import diagnostico_arquitectura as d
from src.modeling import estrategias_horizonte as eh
from src.modeling import experimentos_variables as ev
from src.modeling import fe_hiperparametros as fh
from src.modeling import features_inicio as fi
from src.modeling import hiperparametros as hp
from src.modeling import protocolo_arquitectura as pa
from src.modeling.seguimiento_mlflow import registrar_estudio
from src.utils.paths import (EXPERIMENTOS_FE_HIPERPARAMETROS, FIGURAS_FE_HIPERPARAMETROS, GOLD_MANIFEST,
                             METRICAS_FE_HIPERPARAMETROS, SILVER_EPI_HISTORICO, SILVER_INTEGRADO)

warnings.filterwarnings("ignore", message=".*mismatched devices.*")
pd.set_option("display.precision", 4)
HUELLA = cache.huella_encargo()
CACHE = EXPERIMENTOS_FE_HIPERPARAMETROS / "cache"
DESTINO3 = EXPERIMENTOS_FE_HIPERPARAMETROS / "fase3"
DESTINO3.mkdir(parents=True, exist_ok=True)

panel = cargar_integrado(SILVER_INTEGRADO)
regla = json.loads(GOLD_MANIFEST.read_text(encoding="utf-8"))["regla_brote"]
ins = fh.insumos(panel, regla)
matrices, serie = ins["matrices"], ins["serie"]
variables_inicio = fi.VariablesInicio(panel, serie, pd.read_csv(SILVER_EPI_HISTORICO, dtype={"ubigeo": "string"}), regla)
COLUMNAS = [*eh.GENERICAS, *fi.COLUMNAS["B3"]]
origenes = variables_inicio.origenes(eh.tabla_origenes(matrices[2]), ["B3"], pd.Timestamp("2021-08-28"))
assert not origenes[COLUMNAS].isna().any().any()
print("matriz C3:", COLUMNAS)

C:\Users\Rosa\RufoEsEternoElRegreso\Denguekay\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


matriz C3: ['hist_lag_a', 'hist_lag_b', 'hist_media_4', 'hist_semanas_pos_4', 'semana_epi_seno', 'semana_epi_coseno', 'propia_semanas_desde_ultimo_caso', 'propia_pendiente_4', 'propia_log_casos_temporada_anterior']


In [2]:
pd.Series({k: json.dumps(v, ensure_ascii=False) if isinstance(v, (dict, list)) else v
           for k, v in hp.PREREGISTRO_FASE3.items()}, name="preregistro").to_frame()

,preregistro
fecha,"2026-10-27, antes del estudio (aprobado por Rosa)"
matriz,C3 = base_6 + B3 (elegida en la fase 2c)
base,"{""parametros"": {""n_estimators"": 160, ""max_dept..."
objetivo,log-loss de probabilidad_calibrada (Poisson → ...
espacio,"{""max_depth"": {""tipo"": ""int"", ""min"": 2, ""max"":..."
nota_max_delta_step,"rango 0,3–2 (lineal) en lugar de 0–2: con coun..."
trial_0,"{""parametros"": {""n_estimators"": 160, ""max_dept..."
estudio,"{""muestreador"": ""TPE"", ""semilla_tpe"": 20261027..."
seleccion,las 5 mejores por el objetivo se reentrenan co...
adopcion,gana → se adopta; empata → se queda la base sa...


## Base exacta (C3 con hiperparámetros por defecto, 5 semillas)

In [3]:
base = cache.cargar_o_calcular("fase3_base_C3", lambda: ev.evaluar_y_resumir(matrices, serie, columnas=COLUMNAS,
                                                                             origenes=origenes, hiperparametros=hp.BASE),
                              huella=HUELLA, directorio=CACHE)
objetivo_base = pd.Series({s: hp.objetivo(r)[0] for s, r in base["evaluaciones"].items()}, name="objetivo")
print("objetivo de la base exacta por semilla:", objetivo_base.round(6).to_dict(), "| mediana:", round(objetivo_base.median(), 6))

caché: 'fase3_base_C3' calculada (huella 5a2d16b804ce)
objetivo de la base exacta por semilla: {17: 0.220007, 29: 0.220112, 41: 0.219951, 53: 0.220917, 67: 0.221031} | mediana: 0.220112


## Estudio TPE (100 trials, trial 0 = base con `reg_alpha = 1e-3`)

El estudio se guarda en SQLite (`models/experimentos/fe_hiperparametros/fase3/`), así que si se interrumpe se
reanuda hasta completar 100 trials.

In [4]:
STORAGE = f"sqlite:///{(DESTINO3 / 'optuna_fase3.db').as_posix()}"
estudio = hp.estudio(matrices, serie, origenes, COLUMNAS, STORAGE)
trials = hp.tabla_trials(estudio)
fallidos = trials[trials.estado.ne("COMPLETE")]
print(f"trials: {len(trials)} | completos: {int(trials.estado.eq('COMPLETE').sum())} | fallidos: {len(fallidos)}")
display(fallidos[["numero", "estado"]] if len(fallidos) else "sin trials fallidos")
trials.sort_values("objetivo").head(10)[["numero", "objetivo", "es_base", *[c for c in trials if c.startswith("param_")], "segundos"]]

trials: 100 | completos: 100 | fallidos: 0


'sin trials fallidos'

,numero,objetivo,es_base,param_max_depth,param_min_child_weight,param_learning_rate,param_n_estimators,param_subsample,param_colsample_bytree,param_reg_lambda,param_reg_alpha,param_gamma,param_max_delta_step,segundos
86,86,0.2092,False,3,25.4129,0.0144,1103,0.7966,0.9785,2.1744,0.0141,2.7990,1.7578,26.3675
91,91,0.2095,False,4,41.8181,0.0168,949,0.8473,0.9817,0.5697,0.0103,2.9595,1.8078,29.6748
75,75,0.2097,False,4,42.3760,0.0142,1000,0.9714,0.9484,0.2141,0.0030,2.3914,1.6802,32.2769
82,82,0.2097,False,4,42.7873,0.0164,1012,0.8324,0.9801,0.4037,0.0093,1.9111,1.8204,31.7681
62,62,0.2097,False,3,31.7301,0.0182,1104,0.8839,0.8461,0.2808,0.0170,2.2461,1.9428,27.2000
64,64,0.2098,False,4,44.3040,0.0173,1078,0.8690,0.9171,0.0748,0.0056,2.6852,1.9451,33.4517
65,65,0.2098,False,4,42.9069,0.0168,1087,0.8729,0.9319,0.1772,0.0056,2.7527,1.9676,34.2190
83,83,0.2100,False,4,28.0322,0.0159,1095,0.8343,0.9189,0.8181,0.0280,2.5082,1.9346,34.1354
68,68,0.2101,False,4,27.8640,0.0179,1055,0.8464,0.8986,0.2728,0.0055,2.3479,1.9405,33.3080
59,59,0.2103,False,3,33.0857,0.0208,918,0.7953,0.7690,0.0722,0.0051,2.8767,1.9106,22.8902


In [5]:
trial_0 = trials[trials.es_base].iloc[0]
print(f"trial 0 (base, reg_alpha=1e-3): {trial_0.objetivo:.6f} | base exacta, semilla 17: {objetivo_base[17]:.6f} | "
      f"mejor trial: {trials.objetivo.min():.6f} (n.º {int(trials.loc[trials.objetivo.idxmin(), 'numero'])})")
ruta_curva = d.guardar_figura(hp.figura_curva(trials, float(objetivo_base.median())), FIGURAS_FE_HIPERPARAMETROS,
                              "curva_optimizacion_fase3")
importancia = optuna.importance.get_param_importances(
    estudio, evaluator=optuna.importance.FanovaImportanceEvaluator(seed=hp.SEMILLA_TPE))
ruta_importancia = d.guardar_figura(hp.figura_importancia(importancia), FIGURAS_FE_HIPERPARAMETROS,
                                    "importancia_hiperparametros_fase3")
pd.Series(importancia, name="importancia").round(3)

trial 0 (base, reg_alpha=1e-3): 0.220007 | base exacta, semilla 17: 0.220007 | mejor trial: 0.209188 (n.º 86)


n_estimators        0.675
min_child_weight    0.147
max_delta_step      0.088
max_depth           0.034
gamma               0.020
subsample           0.014
learning_rate       0.013
colsample_bytree    0.005
reg_alpha           0.003
reg_lambda          0.000
Name: importancia, dtype: float64

## Top 5 con 5 semillas y candidata (mejor mediana del objetivo)

In [6]:
mejores = hp.top_k(estudio)
def _calcular_top():
    return {t.number: ev.evaluar_y_resumir(matrices, serie, columnas=COLUMNAS, origenes=origenes,
                                           hiperparametros=t.params) for t in mejores}
top = cache.cargar_o_calcular("fase3_top5", _calcular_top, huella=HUELLA + hp.huella_top(mejores), directorio=CACHE)
tabla_top = pd.DataFrame([{"numero": n, "objetivo_semilla_17_estudio": next(t.value for t in mejores if t.number == n),
                           **{f"objetivo_semilla_{s}": hp.objetivo(e)[0] for s, e in r["evaluaciones"].items()},
                           "objetivo_mediana": float(np.median([hp.objetivo(e)[0] for e in r["evaluaciones"].values()])),
                           "complejidad": hp.complejidad(next(t.params for t in mejores if t.number == n)),
                           **{f"param_{k}": v for k, v in next(t.params for t in mejores if t.number == n).items()}}
                          for n, r in top.items()]).sort_values("objetivo_mediana")
NUMERO_CANDIDATA = int(tabla_top.iloc[0].numero)
PARAMS_CANDIDATA = next(t.params for t in mejores if t.number == NUMERO_CANDIDATA)
candidata = top[NUMERO_CANDIDATA]
print("candidata: trial", NUMERO_CANDIDATA, "| complejidad", hp.complejidad(PARAMS_CANDIDATA), "frente a", hp.complejidad(hp.BASE))
tabla_top

caché: 'fase3_top5' calculada (huella 5a2d16b804ce)


candidata: trial 86 | complejidad 3309 frente a 480


,numero,objetivo_semilla_17_estudio,objetivo_semilla_17,objetivo_semilla_29,objetivo_semilla_41,objetivo_semilla_53,objetivo_semilla_67,objetivo_mediana,complejidad,param_max_depth,param_min_child_weight,param_learning_rate,param_n_estimators,param_subsample,param_colsample_bytree,param_reg_lambda,param_reg_alpha,param_gamma,param_max_delta_step
0,86,0.2092,0.2092,0.2096,0.2095,0.2094,0.2095,0.2095,3309,3,25.4129,0.0144,1103,0.7966,0.9785,2.1744,0.0141,2.7990,1.7578
1,91,0.2095,0.2095,0.2095,0.2095,0.2097,0.2096,0.2095,3796,4,41.8181,0.0168,949,0.8473,0.9817,0.5697,0.0103,2.9595,1.8078
2,75,0.2097,0.2097,0.2097,0.2097,0.2097,0.2097,0.2097,4000,4,42.3760,0.0142,1000,0.9714,0.9484,0.2141,0.0030,2.3914,1.6802
3,82,0.2097,0.2097,0.2097,0.2096,0.2097,0.2099,0.2097,4048,4,42.7873,0.0164,1012,0.8324,0.9801,0.4037,0.0093,1.9111,1.8204
4,62,0.2097,0.2097,0.2101,0.2100,0.2098,0.2101,0.2100,3312,3,31.7301,0.0182,1104,0.8839,0.8461,0.2808,0.0170,2.2461,1.9428


## Regla simétrica: candidata frente a la base exacta (1 comparación; 17 en el encargo)

In [7]:
veredicto_3 = ev.comparar_familia("candidata", candidata["evaluaciones"], candidata["resumen"],
                                  "base", base["evaluaciones"], base["resumen"])
adopcion_3 = hp.adoptar(veredicto_3, PARAMS_CANDIDATA)
display(veredicto_3[["horizonte", "veredicto", "mejora_f1_050", "empeora_f1_050", "mejora_antelacion", "empeora_antelacion",
                     "vetos", "vetos_contra_referencia", "antelacion_diferencia", "antelacion_ic95_inferior",
                     "antelacion_ic95_superior", "mejora_recall_nuevos", "posible_senal_de_temporada"]])
adopcion_3

,horizonte,veredicto,mejora_f1_050,empeora_f1_050,mejora_antelacion,empeora_antelacion,vetos,vetos_contra_referencia,antelacion_diferencia,antelacion_ic95_inferior,antelacion_ic95_superior,mejora_recall_nuevos,posible_senal_de_temporada
0,2,empate,False,False,False,False,None,brier,0.0000,-0.0316,0.0316,False,False
1,3,empate,False,False,False,False,None,brier,0.0105,-0.0211,0.0421,False,False
2,4,empate,False,False,False,False,None,NaN,0.0737,0.0211,0.1368,False,False


{'adoptada': 'base', 'criterio': 'empata y la candidata es más compleja'}

In [8]:
comparacion_3 = pd.concat([ev.tabla_metricas(base["resumen"], "base"), ev.tabla_metricas(candidata["resumen"], "candidata")])
comparacion_3 = comparacion_3[comparacion_3.bloque.isin([*pa.BLOQUES_SELECCION, pa.BLOQUE_CONTRASTE])]
display(comparacion_3.pivot_table(index=["bloque", "horizonte"], columns="variante",
                                  values=["f1_050", "auprc", "brier", "razon_mae", "razon_rmse"]).round(3))
veredicto_3[["horizonte", *[c for c in veredicto_3 if c.startswith(("delta_", "umbral_"))]]].round(4)

auprc            brier           f1_050            \
variante                   base candidata   base candidata   base candidata   
bloque         horizonte                                                      
temporada_2022 2          0.798     0.799  0.052     0.050  0.728     0.724   
               3          0.782     0.788  0.056     0.054  0.612     0.626   
               4          0.739     0.745  0.065     0.063  0.466     0.487   
temporada_2023 2          0.949     0.949  0.061     0.059  0.856     0.864   
               3          0.931     0.931  0.074     0.068  0.829     0.847   
               4          0.918     0.918  0.085     0.075  0.796     0.825   
temporada_2024 2          0.893     0.895  0.076     0.071  0.813     0.812   
               3          0.866     0.875  0.088     0.081  0.786     0.789   
               4          0.821     0.832  0.103     0.096  0.743     0.753   

                         razon_mae           razon_rmse            
variante                      base candidata       base candidata  
bloque         horizonte                                           
temporada_2022 2             0.929     0.976      0.902     0.958  
               3             0.921     0.975      0.900     1.010  
               4             0.947     1.015      0.953     1.132  
temporada_2023 2             1.010     0.950      1.190     1.046  
               3             0.869     0.832      0.986     0.914  
               4             0.796     0.768      0.876     0.833  
temporada_2024 2             1.182     1.351      1.242     1.577  
               3             1.232     1.460      1.245     1.674  
               4             1.300     1.552      1.266     1.722

,horizonte,umbral_f1_050_temporada_2022,umbral_f1_050_temporada_2023,umbral_auprc_temporada_2022,umbral_auprc_temporada_2023,umbral_brier_temporada_2022,umbral_brier_temporada_2023,umbral_razon_mae_temporada_2022,umbral_razon_mae_temporada_2023,delta_f1_050_temporada_2022,delta_f1_050_temporada_2023,delta_auprc_temporada_2022,delta_auprc_temporada_2023,delta_brier_temporada_2022,delta_brier_temporada_2023,delta_razon_mae_temporada_2022,delta_razon_mae_temporada_2023
0,2,0.0117,0.0048,0.0034,0.0007,0.0004,0.0004,0.0150,0.0248,-0.0042,0.0076,0.0015,0.0000,-0.0019,-0.0021,0.0471,-0.0605
1,3,0.0148,0.0049,0.0030,0.0010,0.0009,0.0006,0.0183,0.0143,0.0145,0.0179,0.0056,-0.0002,-0.0020,-0.0054,0.0548,-0.0362
2,4,0.0355,0.0086,0.0038,0.0009,0.0013,0.0007,0.0170,0.0090,0.0216,0.0291,0.0062,-0.0005,-0.0013,-0.0091,0.0681,-0.0282


In [9]:
antelacion_3 = pd.concat({"base": base["resumen"]["antelacion"], "candidata": candidata["resumen"]["antelacion"]},
                         names=["variante"]).reset_index(level=0)
recall_3 = pd.concat({"base": base["resumen"]["recall_por_tipo"], "candidata": candidata["resumen"]["recall_por_tipo"]},
                     names=["variante"]).reset_index(level=0)
display(antelacion_3.pivot_table(index="variante", columns=["horizonte", "bloque"], values="mediana").round(3))
recall_3.query("tipo == 'brote_nuevo'").pivot_table(index="variante", columns=["horizonte", "bloque"], values="recall_mediana").round(3)

horizonte                   2                                               \
bloque    seleccion_2022_2023 temporada_2022 temporada_2023 temporada_2024   
variante                                                                     
base                    0.147          0.132          0.158          0.174   
candidata               0.158          0.132          0.175          0.174   

horizonte                   3                                               \
bloque    seleccion_2022_2023 temporada_2022 temporada_2023 temporada_2024   
variante                                                                     
base                    0.126          0.026          0.193          0.391   
candidata               0.158          0.053          0.228          0.304   

horizonte                   4                                               
bloque    seleccion_2022_2023 temporada_2022 temporada_2023 temporada_2024  
variante                                                                    
base                    0.137            0.0          0.228          0.348  
candidata               0.211            0.0          0.351          0.435

horizonte               2                                               \
bloque    calendario_2025 temporada_2022 temporada_2023 temporada_2024   
variante                                                                 
base                  0.0          0.122          0.128          0.206   
candidata             0.0          0.122          0.140          0.206   

horizonte               3                                               \
bloque    calendario_2025 temporada_2022 temporada_2023 temporada_2024   
variante                                                                 
base                  0.0          0.020          0.140          0.206   
candidata             0.0          0.041          0.174          0.147   

horizonte               4                                               
bloque    calendario_2025 temporada_2022 temporada_2023 temporada_2024  
variante                                                                
base                  0.0            0.0          0.128          0.176  
candidata             0.0            0.0          0.198          0.206

## Robustez (solo lectura): sin las 8 primeras semanas de orígenes de 2017

In [10]:
recorte = ev.recorte_robustez(origenes, "primeras_8_semanas_2017")
o_recortados = origenes.loc[~recorte].reset_index(drop=True)
def _calcular_robustez():
    return {"candidata": ev.evaluar_y_resumir(matrices, serie, columnas=COLUMNAS, origenes=o_recortados, hiperparametros=PARAMS_CANDIDATA),
            "base": ev.evaluar_y_resumir(matrices, serie, columnas=COLUMNAS, origenes=o_recortados, hiperparametros=hp.BASE)}
robustez_3 = cache.cargar_o_calcular("fase3_robustez", _calcular_robustez, huella=HUELLA + hp.huella_top(mejores), directorio=CACHE)
signos_3 = pd.concat({"sin_recorte": ev.signos_por_temporada(candidata["resumen"], base["resumen"]),
                      "sin_8_semanas_2017": ev.signos_por_temporada(robustez_3["candidata"]["resumen"], robustez_3["base"]["resumen"])},
                     names=["recorte"]).reset_index(level=0)
veredicto_rob_3 = ev.comparar_familia("candidata", robustez_3["candidata"]["evaluaciones"], robustez_3["candidata"]["resumen"],
                                      "base", robustez_3["base"]["evaluaciones"], robustez_3["base"]["resumen"])
display(veredicto_rob_3[["horizonte", "veredicto"]])
signos_3.pivot_table(index=["horizonte", "bloque"], columns="recorte", values=["delta_f1_050", "delta_antelacion"]).round(3)

caché: 'fase3_robustez' calculada (huella 5a2d16b804ce)


,horizonte,veredicto
0,2,gana
1,3,gana
2,4,empate


delta_antelacion                   delta_f1_050  \
recorte                  sin_8_semanas_2017 sin_recorte sin_8_semanas_2017   
horizonte bloque                                                             
2         temporada_2022              0.000       0.000              0.032   
          temporada_2023              0.035       0.018              0.008   
3         temporada_2022              0.026       0.026              0.066   
          temporada_2023              0.070       0.035              0.021   
4         temporada_2022              0.000       0.000              0.016   
          temporada_2023              0.088       0.123              0.035   

                                      
recorte                  sin_recorte  
horizonte bloque                      
2         temporada_2022      -0.004  
          temporada_2023       0.008  
3         temporada_2022       0.014  
          temporada_2023       0.018  
4         temporada_2022       0.022  
          temporada_2023       0.029

## Registro (JSON y MLflow)

In [11]:
for nombre, r in (("base", base), ("candidata", candidata)):
    pd.concat(r["predicciones"].values(), ignore_index=True).to_csv(DESTINO3 / f"predicciones_{nombre}.csv.gz", index=False)
trials.to_csv(DESTINO3 / "trials.csv", index=False)
ruta = d.guardar_seccion_metricas(METRICAS_FE_HIPERPARAMETROS, "fase3_hiperparametros", {
    "preregistro": json.loads(json.dumps(hp.PREREGISTRO_FASE3, default=float)),
    "trials": {"total": int(len(trials)), "completos": int(trials.estado.eq("COMPLETE").sum()), "fallidos": int(len(fallidos)),
               "tabla": d.a_json(trials)},
    "base_exacta_objetivo_por_semilla": {int(k): float(v) for k, v in objetivo_base.items()},
    "trial_0_objetivo": float(trial_0.objetivo), "mejor_trial": {"numero": int(trials.loc[trials.objetivo.idxmin(), "numero"]),
                                                                 "objetivo": float(trials.objetivo.min())},
    "importancia_hiperparametros_fanova": {k: float(v) for k, v in importancia.items()},
    "top5_cinco_semillas": d.a_json(tabla_top),
    "candidata": {"trial": NUMERO_CANDIDATA, "parametros": PARAMS_CANDIDATA, "complejidad": hp.complejidad(PARAMS_CANDIDATA),
                  "complejidad_base": hp.complejidad(hp.BASE)},
    "veredicto": d.a_json(veredicto_3), "adopcion": adopcion_3, "comparaciones_acumuladas": 17,
    "metricas_base_y_candidata": d.a_json(comparacion_3), "antelacion": d.a_json(antelacion_3),
    "recall_por_tipo_semana": d.a_json(recall_3),
    "robustez_solo_lectura": {"signos": d.a_json(signos_3), "veredicto_sin_8_semanas_2017": d.a_json(veredicto_rob_3)},
})
print(ruta.relative_to(ROOT))

docs\modeling\metricas\fe_hiperparametros.json


In [12]:
registrar_estudio(
    "fe_hiperparametros_fase3_estudio", trials,
    padre={"params": {**{f"mejor.{k}": v for k, v in estudio.best_params.items()},
                      **{f"candidata.{k}": v for k, v in PARAMS_CANDIDATA.items()},
                      "adopcion": adopcion_3["adoptada"], "criterio": adopcion_3["criterio"]},
           "metricas": {"objetivo_base_exacta_mediana": float(objetivo_base.median()), "objetivo_trial_0": float(trial_0.objetivo),
                        "objetivo_mejor_trial": float(trials.objetivo.min()),
                        "objetivo_candidata_mediana": float(tabla_top.iloc[0].objetivo_mediana),
                        "trials_completos": int(trials.estado.eq("COMPLETE").sum()), "trials_fallidos": int(len(fallidos))}},
    artefactos=[ruta_curva, ruta_importancia, DESTINO3 / "trials.csv"],
    tablas={"top5": tabla_top, "veredicto_frente_a_base": veredicto_3, "metricas_base_y_candidata": comparacion_3,
            "importancia": pd.Series(importancia, name="importancia").reset_index()},
    huella=hp.huella_trials(trials),
    descripcion="Estudio TPE de hiperparámetros de E3·S4_s2 con C3 (HU0007-3) y comparación con la base")

MLflow: ya existe el run d1a4f93b6d274ffc830a61a37f122cba con huella 3200e9ef86f2 en 'denguekay/fe_hiperparametros_fase3_estudio'; no se registra otro


'd1a4f93b6d274ffc830a61a37f122cba'